<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 24 - Almacenamiento Distribuido (HDFS)

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Simular la división de archivos en bloques al estilo HDFS
- Implementar un sistema de replicación con factor configurable
- Simular tolerancia a fallos y recuperación de datos
- Calcular checksums para verificación de integridad

---

## 1. División de archivos en bloques

HDFS divide cada archivo en bloques de tamaño fijo (por defecto 128 MB). Vamos a simular este proceso.

In [ ]:
import hashlib
import random
import os
from collections import defaultdict

def dividir_en_bloques(datos, tam_bloque):
    """Divide datos en bloques de tamaño fijo, como hace HDFS."""
    bloques = []
    for i in range(0, len(datos), tam_bloque):
        bloque = datos[i:i + tam_bloque]
        checksum = hashlib.md5(bloque.encode()).hexdigest()[:8]
        bloques.append({
            'id': f'blk_{i // tam_bloque:04d}',
            'datos': bloque,
            'tamaño': len(bloque),
            'checksum': checksum
        })
    return bloques

contenido = "ABCDEFGHIJ" * 100
print(f"Archivo original: {len(contenido)} bytes")

TAM_BLOQUE = 200
bloques = dividir_en_bloques(contenido, TAM_BLOQUE)

print(f"Tamaño de bloque: {TAM_BLOQUE} bytes")
print(f"Bloques generados: {len(bloques)}\n")

print(f"{'Bloque':<12} {'Tamaño':>8} {'Checksum':<12} {'Primeros 30 chars'}")
print("-" * 65)
for b in bloques:
    print(f"{b['id']:<12} {b['tamaño']:>6} B  {b['checksum']:<12} {b['datos'][:30]}...")

reconstruido = ''.join(b['datos'] for b in bloques)
print(f"\nReconstrucción correcta: {reconstruido == contenido}")

---

## 2. Simulador de HDFS: NameNode y DataNodes

Implementamos las clases principales de HDFS: un NameNode que gestiona metadatos y DataNodes que almacenan bloques.

In [ ]:
class DataNode:
    def __init__(self, nombre, rack):
        self.nombre = nombre
        self.rack = rack
        self.bloques = {}
        self.activo = True
        self.capacidad_total = 1000

    @property
    def espacio_usado(self):
        return sum(len(b) for b in self.bloques.values())

    def almacenar_bloque(self, bloque_id, datos):
        if not self.activo:
            return False
        self.bloques[bloque_id] = datos
        return True

    def leer_bloque(self, bloque_id):
        if not self.activo:
            raise ConnectionError(f"{self.nombre} no está disponible")
        if bloque_id not in self.bloques:
            raise KeyError(f"Bloque {bloque_id} no encontrado en {self.nombre}")
        return self.bloques[bloque_id]

    def __repr__(self):
        estado = "ACTIVO" if self.activo else "CAÍDO"
        return f"{self.nombre} [{self.rack}] ({len(self.bloques)} bloques, {estado})"


class NameNode:
    def __init__(self, factor_replicacion=3):
        self.factor_replicacion = factor_replicacion
        self.datanodes = {}
        self.metadata = {}
        self.block_locations = defaultdict(list)

    def registrar_datanode(self, datanode):
        self.datanodes[datanode.nombre] = datanode

    def _seleccionar_nodos_replica(self, num_replicas):
        """Selecciona DataNodes para réplicas respetando rack awareness."""
        activos = [dn for dn in self.datanodes.values() if dn.activo]
        if len(activos) < num_replicas:
            raise Exception(f"Solo {len(activos)} DataNodes activos, necesito {num_replicas}")

        racks = defaultdict(list)
        for dn in activos:
            racks[dn.rack].append(dn)

        seleccionados = []
        racks_list = list(racks.keys())

        primer_rack = random.choice(racks_list)
        seleccionados.append(random.choice(racks[primer_rack]))

        otros_racks = [r for r in racks_list if r != primer_rack]
        if otros_racks and num_replicas > 1:
            segundo_rack = random.choice(otros_racks)
            nodos_rack2 = [n for n in racks[segundo_rack] if n not in seleccionados]
            if nodos_rack2:
                seleccionados.append(random.choice(nodos_rack2))

        while len(seleccionados) < num_replicas:
            disponibles = [dn for dn in activos if dn not in seleccionados]
            if not disponibles:
                break
            seleccionados.append(random.choice(disponibles))

        return seleccionados

    def escribir_archivo(self, nombre_archivo, bloques):
        """Escribe un archivo distribuyendo bloques con replicación."""
        self.metadata[nombre_archivo] = {
            'bloques': [b['id'] for b in bloques],
            'num_bloques': len(bloques),
            'tamaño_total': sum(b['tamaño'] for b in bloques)
        }

        print(f"Escribiendo '{nombre_archivo}' ({len(bloques)} bloques, RF={self.factor_replicacion})")

        for bloque in bloques:
            nodos = self._seleccionar_nodos_replica(self.factor_replicacion)
            for nodo in nodos:
                nodo.almacenar_bloque(bloque['id'], bloque['datos'])
                self.block_locations[bloque['id']].append(nodo.nombre)
            nodos_str = ', '.join(n.nombre for n in nodos)
            print(f"  {bloque['id']} → [{nodos_str}]")

    def leer_archivo(self, nombre_archivo):
        """Lee un archivo reconstruyéndolo desde los DataNodes."""
        if nombre_archivo not in self.metadata:
            raise FileNotFoundError(f"Archivo '{nombre_archivo}' no encontrado")

        datos_reconstruidos = []
        for bloque_id in self.metadata[nombre_archivo]['bloques']:
            nodos_disponibles = self.block_locations[bloque_id]
            leido = False
            for nodo_nombre in nodos_disponibles:
                nodo = self.datanodes[nodo_nombre]
                try:
                    datos = nodo.leer_bloque(bloque_id)
                    datos_reconstruidos.append(datos)
                    leido = True
                    break
                except (ConnectionError, KeyError):
                    continue
            if not leido:
                raise Exception(f"No se pudo leer el bloque {bloque_id}")

        return ''.join(datos_reconstruidos)

    def estado_cluster(self):
        print("\n" + "=" * 50)
        print("ESTADO DEL CLÚSTER HDFS")
        print("=" * 50)
        for dn in self.datanodes.values():
            print(f"  {dn}")
        print(f"\nArchivos: {len(self.metadata)}")
        print(f"Bloques únicos: {len(self.block_locations)}")
        total_replicas = sum(len(v) for v in self.block_locations.values())
        print(f"Réplicas totales: {total_replicas}")


nn = NameNode(factor_replicacion=3)

for i in range(1, 7):
    rack = f"Rack-{'A' if i <= 3 else 'B'}"
    nn.registrar_datanode(DataNode(f"DN{i}", rack))

contenido = "Datos importantes del sistema Big Data " * 50
bloques = dividir_en_bloques(contenido, 200)

nn.escribir_archivo("datos.csv", bloques)
nn.estado_cluster()

archivo_leido = nn.leer_archivo("datos.csv")
print(f"\nLectura correcta: {archivo_leido == contenido}")

---

## 3. Simulación de tolerancia a fallos

¿Qué ocurre cuando un DataNode cae? HDFS sigue sirviendo datos desde las réplicas.

In [ ]:
print("=" * 50)
print("SIMULACIÓN DE FALLO DE DATANODES")
print("=" * 50)

print("\n--- Situación normal ---")
datos = nn.leer_archivo("datos.csv")
print(f"Lectura OK: {datos == contenido}")

print("\n--- Fallo de DN1 ---")
nn.datanodes["DN1"].activo = False
print(f"DN1 marcado como CAÍDO")
try:
    datos = nn.leer_archivo("datos.csv")
    print(f"Lectura con 1 nodo caído: OK ({datos == contenido})")
except Exception as e:
    print(f"Error: {e}")

print("\n--- Fallo adicional de DN2 ---")
nn.datanodes["DN2"].activo = False
print(f"DN2 marcado como CAÍDO")
try:
    datos = nn.leer_archivo("datos.csv")
    print(f"Lectura con 2 nodos caídos: OK ({datos == contenido})")
except Exception as e:
    print(f"Error: {e}")

nn.estado_cluster()

print("\n--- Recuperación: DN1 vuelve ---")
nn.datanodes["DN1"].activo = True
nn.datanodes["DN2"].activo = True
datos = nn.leer_archivo("datos.csv")
print(f"Lectura tras recuperación: OK ({datos == contenido})")

---

## 4. Verificación de integridad con checksums

HDFS calcula checksums para detectar datos corruptos. Simulemos corrupción y detección.

In [ ]:
import hashlib

def calcular_checksum(datos):
    return hashlib.md5(datos.encode()).hexdigest()

checksums_originales = {}
for bloque_id, nodos in nn.block_locations.items():
    nodo = nn.datanodes[nodos[0]]
    datos_bloque = nodo.bloques[bloque_id]
    checksums_originales[bloque_id] = calcular_checksum(datos_bloque)

print("Checksums originales:")
for bid, cs in list(checksums_originales.items())[:5]:
    print(f"  {bid}: {cs[:16]}...")

bloque_a_corromper = list(nn.block_locations.keys())[2]
nodo_corrupto = nn.datanodes[nn.block_locations[bloque_a_corromper][0]]
nodo_corrupto.bloques[bloque_a_corromper] = "DATOS_CORRUPTOS_!!!!"
print(f"\nBloque {bloque_a_corromper} corrompido en {nodo_corrupto.nombre}")

print("\nVerificando integridad...")
errores = 0
for bloque_id, cs_original in checksums_originales.items():
    for nodo_nombre in nn.block_locations[bloque_id]:
        nodo = nn.datanodes[nodo_nombre]
        if nodo.activo and bloque_id in nodo.bloques:
            cs_actual = calcular_checksum(nodo.bloques[bloque_id])
            if cs_actual != cs_original:
                print(f"  CORRUPCIÓN detectada: {bloque_id} en {nodo_nombre}")
                print(f"    Esperado: {cs_original[:16]}...")
                print(f"    Actual:   {cs_actual[:16]}...")
                errores += 1

if errores == 0:
    print("  Todos los bloques íntegros")
else:
    print(f"\n  {errores} bloque(s) corrupto(s) detectado(s)")
    print("  En HDFS real: se lee de otra réplica y se repara automáticamente")

---

## 5. Análisis de distribución de bloques

Visualicemos cómo se distribuyen los bloques entre los DataNodes y racks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nn2 = NameNode(factor_replicacion=3)
for i in range(1, 7):
    rack = f"Rack-{'A' if i <= 3 else 'B'}"
    nn2.registrar_datanode(DataNode(f"DN{i}", rack))

for i in range(5):
    contenido_i = f"Archivo {i} con datos de ejemplo " * (50 + i * 20)
    bloques_i = dividir_en_bloques(contenido_i, 200)
    nn2.escribir_archivo(f"archivo_{i}.csv", bloques_i)

nodos_nombres = []
nodos_bloques = []
nodos_colores = []

for nombre, dn in sorted(nn2.datanodes.items()):
    nodos_nombres.append(f"{nombre}\n({dn.rack})")
    nodos_bloques.append(len(dn.bloques))
    nodos_colores.append('#3498db' if 'A' in dn.rack else '#e74c3c')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(nodos_nombres, nodos_bloques, color=nodos_colores)
axes[0].set_title('Bloques por DataNode', fontsize=13)
axes[0].set_ylabel('Número de bloques')
axes[0].axhline(y=np.mean(nodos_bloques), color='green', linestyle='--', label=f'Media: {np.mean(nodos_bloques):.1f}')
axes[0].legend()

rack_bloques = defaultdict(int)
for nombre, dn in nn2.datanodes.items():
    rack_bloques[dn.rack] += len(dn.bloques)

axes[1].pie(rack_bloques.values(), labels=rack_bloques.keys(),
            autopct='%1.1f%%', colors=['#3498db', '#e74c3c'],
            startangle=90)
axes[1].set_title('Distribución por Rack', fontsize=13)

plt.tight_layout()
plt.show()

print("Distribución ideal: ~50% por rack con racks equilibrados")

---

## 6. Cálculo de capacidad y coste de almacenamiento

Ejercicio práctico: calcular cuánto espacio real necesitas para almacenar datos en HDFS.

In [ ]:
def calcular_capacidad_hdfs(tam_archivo_gb, tam_bloque_mb=128, factor_replicacion=3):
    """Calcula bloques, réplicas y espacio necesario."""
    tam_archivo_mb = tam_archivo_gb * 1024
    num_bloques = -(-tam_archivo_mb // tam_bloque_mb)  # ceil division
    replicas_totales = num_bloques * factor_replicacion
    espacio_total_gb = (replicas_totales * tam_bloque_mb) / 1024
    overhead = (espacio_total_gb / tam_archivo_gb - 1) * 100

    return {
        'num_bloques': num_bloques,
        'replicas_totales': replicas_totales,
        'espacio_total_gb': espacio_total_gb,
        'overhead_pct': overhead
    }

print(f"{'Archivo':>10} {'Bloque':>10} {'RF':>4} {'Bloques':>10} {'Réplicas':>10} {'Espacio Real':>14} {'Overhead':>10}")
print("-" * 75)

escenarios = [
    (10, 128, 3),
    (100, 128, 3),
    (1000, 128, 3),
    (1000, 256, 3),
    (1000, 128, 2),
    (10000, 128, 3),
]

for tam, bloque, rf in escenarios:
    r = calcular_capacidad_hdfs(tam, bloque, rf)
    print(f"{tam:>8} GB {bloque:>7} MB  RF={rf}  {r['num_bloques']:>9,}  {r['replicas_totales']:>9,}  {r['espacio_total_gb']:>11,.0f} GB  {r['overhead_pct']:>8.0f}%")

print("\nConsejos:")
print("  → RF=3 triplica el espacio, pero es el estándar de producción")
print("  → Bloques más grandes = menos metadata en NameNode")
print("  → El NameNode necesita ~150 bytes de RAM por bloque")

bloques_10pb = calcular_capacidad_hdfs(10_000_000, 128, 3)
ram_namenode_gb = (bloques_10pb['num_bloques'] * 150) / (1024**3)
print(f"\nEjemplo: 10 PB de datos → {bloques_10pb['num_bloques']:,} bloques → ~{ram_namenode_gb:.0f} GB de RAM en NameNode")

---

## Recursos

- [HDFS Architecture Guide](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/HdfsDesign.html)
- [Google File System Paper](https://static.googleusercontent.com/media/research.google.com/en//archive/gfs-sosp2003.pdf)
- [Hadoop: The Definitive Guide](https://www.oreilly.com/library/view/hadoop-the-definitive/9781491901687/)
- [HDFS Federation](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-hdfs/Federation.html)